<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2025-Tutorial-Notebooks/blob/main/exercises/ex4/ex4_ner_bert_given_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


---

### Declaration of Usage of Generative AI
In this work, Generative AI was used to post-edit the wording of **some** written paragraphs and comments, including the ones in the final report, to improve their clarity and style, as English is not the native language of any of the group members. The underlying ideas and content remain entirely our own. In terms of coding, AI assistance was used **solely** for debugging purposes when exceptions occurred in the code. **No** generative AI tools were used as a learning assistant during the completion of this exercise.

---



In [ ]:
import os
import torch
import random
import numpy as np


def seed_everything(seed: int = 1) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    os.environ["PYTHONHASHSEED"] = str(seed)

    print(f"[Seeded everything with seed={seed}]")

seed_everything(1)

[Seeded everything with seed=1]


# Load and prepare the required data:

In [ ]:
!pip install datasets

In [ ]:
# We have selected the German language for this exercise. Its code is "de," and it is supported.
chosen_language_code = "de"

In [ ]:
import datasets

# NOTE: If the maximum sequence length exceeds the model's maximum
# sequence length, you need to make adjustments (for example, when
# choosing 'en')
test_set = datasets.load_dataset("unimelb-nlp/wikiann", chosen_language_code, split="test[:2000]")
train_set1000 = datasets.load_dataset("unimelb-nlp/wikiann", chosen_language_code, split="train[:1000]")
train_set3000 = datasets.load_dataset("unimelb-nlp/wikiann", chosen_language_code, split="train[:3000]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

de/validation-00000-of-00001.parquet:   0%|          | 0.00/835k [00:00<?, ?B/s]

de/test-00000-of-00001.parquet:   0%|          | 0.00/832k [00:00<?, ?B/s]

de/train-00000-of-00001.parquet:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

**NOTE: Make sure that there are indeed as many data points in the above sets**

In [ ]:
print(train_set1000)
print(train_set3000)
print(test_set)

Dataset({
    features: ['tokens', 'ner_tags', 'langs', 'spans'],
    num_rows: 1000
})
Dataset({
    features: ['tokens', 'ner_tags', 'langs', 'spans'],
    num_rows: 3000
})
Dataset({
    features: ['tokens', 'ner_tags', 'langs', 'spans'],
    num_rows: 2000
})


In [ ]:
ner_tags = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

### Inspect and Describe the Data, including Average and Maximum Input length (in tokens):

In [ ]:
import pandas as pd
from typing import Dict, List
from torch.utils.data import Dataset

# Define the function which computes the following statistics in the dataset:
# - 'avg_length': The average number of tokens per sample.
# - 'std_length': The standard deviation of the number of tokens per sample.
# - 'min_length': The shortest number of tokens per sample in the dataset.
# - 'max_length': The longest number of tokens per sample in the dataset.
# - 'avg_unique': The average number of unique tokens per sample.
# - 'std_unique': The standard deviation of unique token counts.
# - 'min_unique': The smallest number of unique tokens in any sample.
# - 'max_unique': The largest number of unique tokens in any sample.
def dataset_stats(data: Dataset,
                  dataset_name: str) -> Dict[str, float]:
  stats = {}
  lengths = []
  num_unique = []

  # Define a dictionary with reverse NER tags
  ner_tags = {0: "O",
              1: "B-PER",
              2: "I-PER",
              3: "B-ORG",
              4: "I-ORG",
              5: "B-LOC",
              6: "I-LOC"}

  # Sample 3 sentences from the dataset
  random_sample = np.random.choice(data, 3, replace=False)
  print(f'Random sample from {dataset_name}:')
  for sample in random_sample:
    line1 = ''
    line2 = ''
    for word, label in zip(sample['tokens'], sample['ner_tags']):
      text_label = ner_tags[label]
      max_length = max(len(word), len(text_label))
      line1 += word + " " * (max_length - len(word) + 1)
      line2 += text_label + " " * (max_length - len(text_label) + 1)
    print(line1)
    print(line2)
    print('-' * 100)
  print('\n')

  # Loop through the dataset and collect token lengths and unique token counts
  for sample in data:
    lengths.append(len(sample['tokens']))
    num_unique.append(len(set(sample['tokens'])))

  # Calculate statistics based on lengths and unique token counts
  stats['avg_length'] = np.mean(lengths)
  stats['std_length'] = np.std(lengths)
  stats['min_length'] = min(lengths)
  stats['max_length'] = max(lengths)
  stats['avg_unique'] = np.mean(num_unique)
  stats['std_unique'] = np.std(num_unique)
  stats['min_unique'] = min(num_unique)
  stats['max_unique'] = max(num_unique)

  return stats

# Extract statistics for each training dataset
stats_1000 = dataset_stats(train_set1000, '1000 sentences')
stats_3000 = dataset_stats(train_set3000, '3000 sentences')

# Create a dataframe
print('Dataset statistics:')
pd.DataFrame([stats_1000, stats_3000], index=['1000', '3000'])

Random sample from 1000 sentences:
1995 - Sérgio Godinho 
O    O B-PER  I-PER   
----------------------------------------------------------------------------------------------------
Dareios II    .     
B-PER   I-PER I-PER 
----------------------------------------------------------------------------------------------------
Stadio Pierluigi Penzo – Venedig 
B-ORG  I-ORG     I-ORG O B-LOC   
----------------------------------------------------------------------------------------------------


Random sample from 3000 sentences:
John  Carisi und Joe   Mooney . 
B-PER I-PER  O   B-PER I-PER  O 
----------------------------------------------------------------------------------------------------
Gewinner der Trophäe wurde der Racing Club  de    Paris . 
O        O   O       O     O   B-ORG  I-ORG I-ORG I-ORG O 
----------------------------------------------------------------------------------------------------
Gleichzeitig wechselte man erneut den Ausrüster , diesmal bekam Nike  den Zuschlag 

,avg_length,std_length,min_length,max_length,avg_unique,std_unique,min_unique,max_unique
1000,9.767000,6.470913,1,76,9.108,6.064185,1,68
3000,9.738667,6.246362,1,76,9.068,5.927398,1,68


 ✅ We observe that both datasets exhibit nearly identical statistical properties across all measured metrics. The average number of unique tokens per sentence is very close to the overall average sentence length, suggesting that most sentences consist predominantly of non-repeating words. Since the first dataset is a subset of the second and their distributions align closely, we can conclude that the global dataset sampling process was stable and consistent. There are no statistically significant differences between the two datasets, indicating that the larger dataset preserves the same structural and linguistic characteristics as the smaller one.

📝❓Why do you need to be aware of the longest input length within your dataset? Which parameter of the model dictates this?

 ✅ Being aware of the longest input length within our dataset is crucial because transformer models like BERT have a fixed maximum sequence length. For BERT, this limit is 512 tokens, meaning it cannot process input sequences longer than that. If any sentence in the dataset exceeds this length, it must be truncated or split into smaller segments to fit within the model's capacity. The parameter that controls this behavior during preprocessing is the `max_length` in the tokenizer.

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-german-cased')

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/255k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/485k [00:00<?, ?B/s]

In [ ]:
# Set the maximum sequence length to the value obtained above, the tokenization will
# slightly increase the number of tokens compared to the original text (due to subword
# splitting and special tokens).
max_sequence_length = stats_3000['max_length']
print(f'The maximum sequence length: {max_sequence_length}')

The maximum sequence length: 76


📝❓The dataset is split into words, and the assigned labels are for words. How should we deal with labels **after** tokenization? NOTE: Each word may be split into one or multiple tokens by the tokenizer.

In [ ]:
# Print an example of the tokenization result
inputs = tokenizer(train_set1000[0]['tokens'],
                   return_tensors='pt',
                   padding='max_length',
                   truncation=True,
                   max_length=max_sequence_length,
                   is_split_into_words=True)
print(inputs.tokens())

['[CLS]', 'als', 'Teil', 'der', 'Sav', '##oy', '##er', 'Vor', '##alp', '##en', 'im', 'Osten', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


 ✅ When using a BERT tokenizer on our dataset, which labeled at the word level, each word may be split into one or more subword tokens. Since the labels correspond to full words, we must correctly align them with the tokenized output. Our approach is to assign the original word's label only to the first subword of that word. All continuation subwords (those that start with “##”) and all special tokens such as [CLS], [SEP], and [PAD] are assigned the label -100. This ensures that these tokens are ignored by BERT's loss function during training, i.e. the model learns and updates its parameters only for the tokens that correspond to actual words in the dataset.

In [ ]:
# Implement the function which encodes the inputs and aligns the labels
def encode_and_align_labels(example: Dict[str, List],
                            tokenizer: AutoTokenizer = tokenizer,
                            max_sequence_length: int = max_sequence_length) -> Dict[str, List]:
    """Tokenizes the input tokens and aligns the word-level NER labels with the tokenized output."""

    # Tokenize the input example, telling the tokenizer that it is already split into words.
    # We pad/truncate to a fixed max length for the model compatibility.
    inputs = tokenizer(example['tokens'],
                       padding='max_length',
                       truncation=True,
                       max_length=max_sequence_length,
                       is_split_into_words=True)

    # Original word-level labels (same length as number of words in example["tokens"])
    labels = example['ner_tags']

    # Map each produced token back to its originating word index (or None for specials/padding).
    word_ids = inputs.word_ids()

    # Build token-level labels:
    # - For special tokens ([CLS], [SEP], [PAD]) => -100 (ignored by the loss)
    # - For the first subword of a word          => its word-level label
    # - For continuation subwords                => -100 (ignored by loss)
    new_labels = []
    prev_id = None
    for cur_id in word_ids:
        if cur_id is None:
            new_labels.append(-100)
        elif cur_id != prev_id:
            new_labels.append(labels[cur_id])
        else:
            new_labels.append(-100)
        prev_id = cur_id

    # Append aligned labels to the model inputs
    inputs['labels'] = new_labels

    # Convert the lists in the dictionary to 1D tensors of shape (L,) per example,
    # since the trainer will stack them to (B, L) later
    for key, val in inputs.items():
      if isinstance(val, list):
        inputs[key] = torch.tensor(val)

    return inputs

In [ ]:
# Encode the two training sets and the test set by applying the function above to each dataset entry
encoded_test_set = test_set.map(encode_and_align_labels, batched=False)
encoded_train_set1000 = train_set1000.map(encode_and_align_labels, batched=False)
encoded_train_set3000 = train_set3000.map(encode_and_align_labels, batched=False)

# Set format for PyTorch
encoded_test_set.set_format(
    type="torch",
    columns=["input_ids", "token_type_ids", "attention_mask", "labels"]
)
encoded_train_set1000.set_format(
	type="torch",
	columns=["input_ids", "token_type_ids", "attention_mask", "labels"]
)
encoded_train_set3000.set_format(
	type="torch",
	columns=["input_ids", "token_type_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
# Check out how the training sets are encoded
for key, val in encoded_train_set1000[0].items():
    print(f'{key}: {val.size()}')

input_ids: torch.Size([76])
token_type_ids: torch.Size([76])
attention_mask: torch.Size([76])
labels: torch.Size([76])


Example of how your output could look like.

input_ids: torch.Size([???])

token_type_ids: torch.Size([???])

attention_mask: torch.Size([???])

labels: torch.Size([???])

📝❓What value should replace the three question marks in your print? Should this be the same for all samples? Why/Why not?

✅ The value that replaces the three question marks is 76 (`max_sequence_length` defined earlier), which is well within the maximum input length supported by BERT. Any resulting sequence longer than 76 tokens was truncated. Yes, the input length should be consistent across all samples to ensure that tensors within each batch have identical dimensions, allowing for efficient batching and parallel computation during training. In our dataset, the longest sequence at the word level contained 76 tokens, while the average length of the sentence was approximately 10 tokens with a standard deviation of about 6.5. Given this, the choice of 76 as the maximum sequence length leaves substantial room to accommodate additional subwords created during tokenization as well as the special tokens added by the model, without any loss of information. As a result, the most of the original content is preserved, and the data is standardized for consistent model input.

# Training

## Training Utils

In [ ]:
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments
import os
os.environ["WANDB_MODE"] = "disabled"

### Complete the following, reusable functions:

In [ ]:
from sklearn.metrics import f1_score
from transformers.trainer_utils import PredictionOutput

def compute_metrics(preds: PredictionOutput) -> Dict[str, float]:
    """
    Compute macro and micro F1 scores from PredictionOutput

    Args:
        preds: transformers.trainer_utils.PredictionOutput

    Returns:
        dict with macro_f1 and micro_f1 scores
    """

    # Extract the ground truth and predictions from the model output
    y_true = preds.label_ids
    y_pred = preds.predictions.argmax(axis=-1)

    # Ignore special/pad/continuation positions labeled as -100
    mask = y_true != -100
    y_true_flat = y_true[mask].ravel()
    y_pred_flat = y_pred[mask].ravel()

    # Return 0.0 if the sequence contained only special tokens
    if y_true_flat.size == 0:
      return {'f1_macro': 0.0,
              'f1_micro': 0.0}

    return {'f1_macro': f1_score(y_true_flat, y_pred_flat, average='macro'),
            'f1_micro': f1_score(y_true_flat, y_pred_flat, average='micro')}

In [ ]:
from transformers import PreTrainedModel

def freeze_weights(model: PreTrainedModel) -> PreTrainedModel:
    """Freeze the weights for a given model.

    Args:
        model: transformers.PreTrainedModel

    Returns:
			  model: transformers.PreTrainedModel
    """

    # Disable gradient computation
    for p in model.parameters():
      p.requires_grad = False

    # Switch to evaluation mode
    model.eval()

    return model

## Variant 1: 1000 sentences, no frozen weights

### Initialise your model and set up your training arguments:

In [ ]:
# Load a token classification model (for NER) based on a cased German BERT.
# Here we do not freeze the backbone weights
model_var1 = AutoModelForTokenClassification.from_pretrained('bert-base-german-cased',
                                                             num_labels=len(ner_tags))


# Define training/eval hyperparameters and I/O locations
train_args_var1 = TrainingArguments(num_train_epochs=4,
                                    per_device_train_batch_size=16,
                                    per_device_eval_batch_size=16,
                                    output_dir='results',
                                    logging_dir='logs',
                                    no_cuda=False)

# Create a Trainer to handle training loop, evaluation, logging, and checkpointing.
# Here we use the training dataset of 1000 sentences
trainer_var1 = Trainer(model=model_var1,
                       tokenizer=tokenizer,
                       args=train_args_var1,
                       train_dataset=encoded_train_set1000,
                       eval_dataset=encoded_test_set,
                       compute_metrics=compute_metrics)

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-german-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1426810468.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_var1 = Trainer(model=model_var1,


📝❓When initializing the BertForTokenClassification-class with BERT-base you should get a warning message. Explain why you get this message.

 ✅ There are two warnings. The **first** message appears when we initialize `BertForTokenClassification`, because the original BERT checkpoint does not include a token classification layer. The base model only contains the pretrained encoder weights, while the classification head is specific to downstream applications. Since these layers (`classifier.weight` and `classifier.bias`) don't exist in the pretrained checkpoint, they are randomly initialized when we create the model. The warning simply reminds us that this new classification head must be finetuned on labeled data to learn task-specific weights before using the model for inference. The second message is a FutureWarning that `tokenizer` is deprecated in `Trainer` and will be removed in version 5.0.0.


### Train your Model ⚡ GPU 2-3 mins:

In [ ]:
# Train the model on the training dataset of 1000 sentences and store training results
train_output_var1 = trainer_var1.train()

# Evaluate the trained model on the evaluation dataset and store metrics
eval_output_var1 = trainer_var1.evaluate()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


Step,Training Loss


📝❓ Is there a challenge when evaluating the predictions of your model? Why is this challenge present and how do you plan to deal with it?

Hint: Look at the lengths

 ✅ A key challenge when evaluating the model's predictions arises from handling sequences of varying lengths. Because BERT requires uniform input lengths, all sentences are either padded or truncated to match `max_sequence_length`. As a result, the model also generates predictions for padded tokens, which do not correspond to real words. Including these in evaluation would distort the F1 scores. Thus, our `compute_metrics` applies a mask that ignores all positions labeled as `-100`, effectively excluding special, padding, and continuation tokens. This ensures that F1 scores are computed only on meaningful tokens, providing an accurate assessment of the model's true performance.

### Compute Metrics/Performance of your model:

To avoid rerunning, please also print the metrics of each model that completed training

In [ ]:
print('Evaluation metrics of Variant 1: 1000 sentences, no frozen weights:')
print(eval_output_var1)

Evaluation metrics of Variant 1: 1000 sentences, no frozen weights:
{'eval_loss': 0.24234747886657715, 'eval_f1_macro': 0.8514916387346582, 'eval_f1_micro': 0.9405138852841941, 'eval_runtime': 9.0502, 'eval_samples_per_second': 220.99, 'eval_steps_per_second': 13.812, 'epoch': 4.0}


## Variant 2: 3000 sentences, no frozen weights

In [ ]:
# Clear cached GPU memory to free up VRAM
torch.cuda.empty_cache()

In [ ]:
# Load a token classification model (for NER) based on a cased German BERT.
# Here we do not freeze the backbone weights
model_var2 = AutoModelForTokenClassification.from_pretrained('bert-base-german-cased',
                                                             num_labels=len(ner_tags))


# Define training/eval hyperparameters and I/O locations
train_args_var2 = TrainingArguments(num_train_epochs=4,
                                    per_device_train_batch_size=64,
                                    per_device_eval_batch_size=64,
                                    output_dir='results',
                                    logging_dir='logs',
                                    no_cuda=False)

# Create a Trainer to handle training loop, evaluation, logging, and checkpointing.
# Here we use the training dataset of 3000 sentences
trainer_var2 = Trainer(model=model_var2,
                       tokenizer=tokenizer,
                       args=train_args_var2,
                       train_dataset=encoded_train_set3000,
                       eval_dataset=encoded_test_set,
                       compute_metrics=compute_metrics)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-german-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3565474276.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_var2 = Trainer(model=model_var2,


In [ ]:
# Train the model on the training dataset of 3000 sentences and store training results
train_output_var2 = trainer_var2.train()

# Evaluate the trained model on the evaluation dataset and store metrics
eval_output_var2 = trainer_var2.evaluate()

Step,Training Loss


In [ ]:
print('Evaluation metrics of Variant 2: 3000 sentences, no frozen weights:')
print(eval_output_var2)

Evaluation metrics of Variant 2: 3000 sentences, no frozen weights:
{'eval_loss': 0.1785379946231842, 'eval_f1_macro': 0.8824690932668809, 'eval_f1_micro': 0.9530236179600311, 'eval_runtime': 8.5944, 'eval_samples_per_second': 232.71, 'eval_steps_per_second': 3.723, 'epoch': 4.0}


## Variant 3: 1000 sentences, frozen weights

In [ ]:
# Clear cached GPU memory to free up VRAM
torch.cuda.empty_cache()

In [ ]:
# Load a token classification model (for NER) based on a cased German BERT
model_var3 = AutoModelForTokenClassification.from_pretrained('bert-base-german-cased',
                                                             num_labels=len(ner_tags))

# Freeze the weights of the backbone (model.bert)
freeze_weights(model_var3.bert)

# Define training/eval hyperparameters and I/O locations
train_args_var3 = TrainingArguments(num_train_epochs=4,
                                    per_device_train_batch_size=16,
                                    per_device_eval_batch_size=16,
                                    output_dir='results',
                                    logging_dir='logs',
                                    no_cuda=False)

# Create a Trainer to handle training loop, evaluation, logging, and checkpointing.
# Here we use the training dataset of 1000 sentences
trainer_var3 = Trainer(model=model_var3,
                       tokenizer=tokenizer,
                       args=train_args_var3,
                       train_dataset=encoded_train_set1000,
                       eval_dataset=encoded_test_set,
                       compute_metrics=compute_metrics)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-german-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2340655882.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_var3 = Trainer(model=model_var3,


In [ ]:
# Train the classification head on the training dataset of 1000 sentences and store training results
train_output_var3 = trainer_var3.train()

# Evaluate the trained model on the evaluation dataset and store metrics
eval_output_var3 = trainer_var3.evaluate()

Step,Training Loss


In [ ]:
print('Evaluation metrics of Variant 3: 1000 sentences, frozen weights:')
print(eval_output_var3)

Evaluation metrics of Variant 3: 1000 sentences, frozen weights:
{'eval_loss': 1.215296745300293, 'eval_f1_macro': 0.13763137108314696, 'eval_f1_micro': 0.7003893070334805, 'eval_runtime': 11.096, 'eval_samples_per_second': 180.246, 'eval_steps_per_second': 11.265, 'epoch': 4.0}


## Variant 4: 3000 sentences, frozen weights

In [ ]:
# Clear cached GPU memory to free up VRAM
torch.cuda.empty_cache()

In [ ]:
# Load a token classification model (for NER) based on a cased German BERT
model_var4 = AutoModelForTokenClassification.from_pretrained('bert-base-german-cased',
                                                             num_labels=len(ner_tags))

# Freeze the weights of the backbone (model.bert)
freeze_weights(model_var4.bert)

# Define training/eval hyperparameters and I/O locations
train_args_var4 = TrainingArguments(num_train_epochs=4,
                                    per_device_train_batch_size=64,
                                    per_device_eval_batch_size=64,
                                    output_dir='results',
                                    logging_dir='logs',
                                    no_cuda=False)

# Create a Trainer to handle training loop, evaluation, logging, and checkpointing.
# Here we use the training dataset of 3000 sentences
trainer_var4 = Trainer(model=model_var4,
                       tokenizer=tokenizer,
                       args=train_args_var4,
                       train_dataset=encoded_train_set3000,
                       eval_dataset=encoded_test_set,
                       compute_metrics=compute_metrics)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-german-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-950312447.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_var4 = Trainer(model=model_var4,


In [ ]:
# Train the classification head on the training dataset of 3000 sentences and store training results
train_output_var4 = trainer_var4.train()

# Evaluate the trained model on the evaluation dataset and store metrics
eval_output_var4 = trainer_var4.evaluate()

Step,Training Loss


In [ ]:
print('Evaluation metrics of Variant 4: 3000 sentences, frozen weights:')
print(eval_output_var4)

Evaluation metrics of Variant 4: 3000 sentences, frozen weights:
{'eval_loss': 1.2620327472686768, 'eval_f1_macro': 0.14297738222713233, 'eval_f1_micro': 0.6976382039968856, 'eval_runtime': 8.3897, 'eval_samples_per_second': 238.386, 'eval_steps_per_second': 3.814, 'epoch': 4.0}


# Report




📝❓ Template:

Summary of Performance of the four Model Variants

1. Whole Model finetuning, 1000 samples:
2. Whole Model finetuning, 3000 samples:
3. Frozen Backbone, 1000 samples:
4. Frozen Backbone 3000 samples:

In [ ]:
print('Performance of the four Model Variants:')
pd.DataFrame([eval_output_var1, eval_output_var2, eval_output_var3, eval_output_var4],
             index=[f'Variant {i}' for i in range(1, 5)])

Performance of the four Model Variants:


,eval_loss,eval_f1_macro,eval_f1_micro,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
Variant 1,0.242347,0.851492,0.940514,9.0502,220.990,13.812,4.0
Variant 2,0.178538,0.882469,0.953024,8.5944,232.710,3.723,4.0
Variant 3,1.215297,0.137631,0.700389,11.0960,180.246,11.265,4.0
Variant 4,1.262033,0.142977,0.697638,8.3897,238.386,3.814,4.0


📝❓ When we freeze the transformer backbone weights, which weights are being tuned during fine-tuning?

 ✅ When we freeze the BERT backbone, all parameters in the embedding layer and transformer encoder blocks, including attention, feed-forward, and layer-norm weights, are frozen and remain unchanged during training. The only trainable part is the task-specific classification head, typically a small `nn.Linear` layer that maps the final hidden states to label logits. This allows the model to adapt the output layer to the downstream task while keeping the pretrained BERT representations fixed.

📝❓ Are there differences between f1-micro and f1-macro score? If so, why?

✅ Yes, there are important differences:

* **F1-micro:** Computes metrics globally by aggregating all true positives, false negatives, and false positives across classes before calculating the F1 score.
* **F1-macro:** Computes the F1 score individually for each class and then takes their unweighted mean, giving equal importance to all classes, regardless of their frequency.

F1-micro reflects the model's overall performance, while F1-macro is better suited for imbalanced datasets, as it highlights performance on rare or underrepresented classes.



📝❓ Is it better to freeze or not to freeze the transformer backbone weights? Hypothesize why

✅ Freezing the backbone reduces the number of trainable parameters, which lowers GPU memory usage and speeds up training, but it also limits the model's ability to adapt to the new task. This approach can be useful when working with small datasets, where fully finetuning the model might lead to overfitting. In contrast, keeping the backbone unfrozen allows BERT to adjust its contextual representations to the specific task and language. In our case, for NER on the WikiANN dataset, **not freezing** the backbone performs better, as it better aligns embeddings with labels.


📝❓ Write your lab report here addressing all questions in the notebook

✅ In this notebook, we explored token level classification, specifically named entity recognition using a BERT architecture on the WikiANN dataset, focusing on German as the target language.

Both the 1000 and 3000-sentence subsets used in this task exhibited nearly identical statistical properties, indicating that the larger dataset maintains the same structural and linguistic characteristics as the smaller one. Since transformer models such as BERT have a fixed maximum sequence length of 512 tokens, we examined sentence lengths and found that all were well below this threshold. Consequently, we set the tokenizer to truncate sequences exceeding the computed `max_sequence_length` of 76 tokens after tokenization.

Because the dataset is word-labeled while BERT operates on subword tokens, label alignment was required. We assigned each word's label to its first subword token and used `-100` for continuation subwords and special tokens (`[CLS]`, `[SEP]`, `[PAD]`) to ensure they were ignored by the loss function and evaluation metrics. This alignment allowed the model to learn only from meaningful tokens.

Each model was trained for 4 epochs, with a batch size of 16 for the smaller and 64 for the larger dataset. Performance was evaluated using F1-micro and F1-macro scores.

The **takeaways** from the results are that full fine-tuning yields significant performance gains in named entity recognition, achieving strong F1 scores. Increasing the dataset size under full fine-tuning further improves both micro and macro metrics, highlighting the benefits of additional training data. However, freezing the backbone, while reducing training time and memory usage, limits the model's adaptability and results in lower overall performance.
